# Phase 5 — Hybrid Movie Recommendation System

## Objective

Combine Content-Based Filtering and Collaborative Filtering into a single Hybrid Recommendation System.

### Hybrid Score

Hybrid Score = 0.50 × Content Score + 0.50 × Collaborative Score

Both components are normalized before combination.

The hybrid approach provides both movie-level similarity and user-level personalization.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split


In [ ]:
movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")

movies["title"] = movies["title"].fillna("Unknown Movie").astype(str)
movies["genres"] = movies["genres"].fillna("Unknown").astype(str)

ratings = ratings[
    ["userId", "movieId", "rating"]
].dropna()

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)


## Content-Based Component

In [ ]:
movies["genre_text"] = movies["genres"].str.replace(
    "|", " ", regex=False
)

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["genre_text"])

print("TF-IDF Shape:", tfidf_matrix.shape)


In [ ]:
similarity_matrix = cosine_similarity(tfidf_matrix)

title_to_index = pd.Series(
    movies.index,
    index=movies["title"]
).drop_duplicates()

print("Similarity Matrix Shape:", similarity_matrix.shape)


## Collaborative Filtering Component

In [ ]:
reader = Reader(
    rating_scale=(
        float(ratings["rating"].min()),
        float(ratings["rating"].max())
    )
)

data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

trainset, testset = train_test_split(
    data,
    test_size=0.20,
    random_state=42
)


In [ ]:
model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

model.fit(trainset)

print("Collaborative model trained.")


In [ ]:
def get_rated_movie_ids(user_id):
    return set(
        ratings.loc[
            ratings["userId"] == user_id,
            "movieId"
        ]
    )


## Hybrid Recommendation Function

In [ ]:
def hybrid_recommend(user_id, movie_title, n=10):
    if movie_title not in title_to_index:
        return pd.DataFrame()

    movie_index = title_to_index[movie_title]
    similarity_scores = similarity_matrix[movie_index]

    candidates = movies.copy()
    candidates["content_score"] = similarity_scores

    rated_movie_ids = get_rated_movie_ids(user_id)

    candidates = candidates[
        ~candidates["movieId"].isin(rated_movie_ids)
    ].copy()

    candidates = candidates[
        candidates["title"] != movie_title
    ].copy()

    candidates["predicted_rating"] = candidates["movieId"].apply(
        lambda movie_id: float(
            model.predict(
                int(user_id),
                int(movie_id)
            ).est
        )
    )

    # Normalize content score to 0-1
    content_min = candidates["content_score"].min()
    content_max = candidates["content_score"].max()

    if content_max > content_min:
        candidates["content_norm"] = (
            (candidates["content_score"] - content_min)
            / (content_max - content_min)
        )
    else:
        candidates["content_norm"] = 0.0

    # Normalize predicted rating to 0-1
    rating_min = candidates["predicted_rating"].min()
    rating_max = candidates["predicted_rating"].max()

    if rating_max > rating_min:
        candidates["rating_norm"] = (
            (candidates["predicted_rating"] - rating_min)
            / (rating_max - rating_min)
        )
    else:
        candidates["rating_norm"] = 0.0

    # 50% content + 50% collaborative
    candidates["hybrid_score"] = (
        0.50 * candidates["content_norm"]
        + 0.50 * candidates["rating_norm"]
    )

    recommendations = candidates.sort_values(
        "hybrid_score",
        ascending=False
    ).head(n)

    return recommendations[
        [
            "movieId",
            "title",
            "genres",
            "predicted_rating",
            "content_score",
            "hybrid_score"
        ]
    ].reset_index(drop=True)


In [ ]:
selected_user = int(ratings["userId"].iloc[0])

movies[
    movies["title"].str.contains(
        "Toy Story",
        case=False,
        na=False,
        regex=False
    )
][["movieId", "title", "genres"]].head(10)


In [ ]:
hybrid_results = hybrid_recommend(
    selected_user,
    "Toy Story (1995)",
    10
)

hybrid_results


In [ ]:
hybrid_results[
    [
        "title",
        "genres",
        "predicted_rating",
        "content_score",
        "hybrid_score"
    ]
]


In [ ]:
print("Minimum Hybrid Score:", hybrid_results["hybrid_score"].min())
print("Maximum Hybrid Score:", hybrid_results["hybrid_score"].max())


In [ ]:
duplicate_count = hybrid_results["movieId"].duplicated().sum()

print("Duplicate Recommendations:", duplicate_count)


In [ ]:
rated_ids = get_rated_movie_ids(selected_user)
recommended_ids = set(hybrid_results["movieId"])

overlap = recommended_ids & rated_ids

print("Already-rated movies recommended:", len(overlap))
print("Overlap:", overlap)


## Hybrid Recommendation Architecture

Selected User + Selected Movie

→ Content-Based TF-IDF and Cosine Similarity

→ Collaborative SVD Predicted Rating

→ Normalize both components

→ Apply 50% + 50% weighted combination

→ Rank candidates

→ Return final recommendations


## Interpretation

The Hybrid Recommendation System combines movie similarity with personalized user preference prediction.

Content similarity identifies movies related to the selected title, while SVD estimates how much the selected user may like each candidate.

The normalized components are combined using equal weights, producing the final hybrid ranking.


# Phase 5 Conclusion

The Hybrid Movie Recommendation System was successfully implemented.

It combines Content-Based Filtering and Collaborative Filtering to provide recommendations that consider both the characteristics of the selected movie and the selected user's learned preferences.
